# NAC Chip Verification v2

This notebook recreates NAC chips from raw NAC source GeoTIFFs and compares them against provided processed NAC chips. The goal is to check whether the provided chips look altered, blurred, resampled differently, or otherwise processed unexpectedly.

The important difference from `chip_NAC.ipynb` is that this notebook does **not** require the provided chip product ID to match a raw NAC filename. It first queries the raw NAC tile index by lat/lon overlap, then recreates a chip from the overlapping raw NAC source imagery.


## Setup


In [ ]:
import os
os.environ["MALLOC_CONF"] = "oversize_threshold:1,background_thread:true,metadata_thp:auto"

from contextlib import redirect_stdout
from pathlib import Path
import sys
import time
import json
import re
import shutil

import numpy as np
import pandas as pd
import rasterio
import rioxarray as rxr
from rioxarray.merge import merge_arrays
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from tqdm import tqdm
from shapely.geometry import box
from shapely.ops import transform as shapely_transform
from pyproj import CRS, Transformer
from osgeo import gdal

gdal.UseExceptions()



In [ ]:
# Resolve repo root when running from notebooks/toy_model.
repo_root = Path.cwd().parent.parent
repo_root = Path(str(repo_root).replace("/panfs/ccds02/nobackup", "/explore/nobackup"))
if not (repo_root / "model").exists():
    raise FileNotFoundError(
        "Cannot find lfm/model directory. Run this notebook from lfm/notebooks/toy_model."
    )

sys.path.insert(0, str(repo_root.parent))

from lfm.model.Pipeline import Pipeline
from lfm.model.TmsIntersector import TmsIntersector
from lfm.model.TmsTileDef import TmsTileDef
from lfm.model.chip_making.chip_utils import get_worker_logger
from lfm.lfm.toy_model.all_tasks.all_utils import prepare_output_dir

logger = get_worker_logger()
print(f"repo_root: {repo_root}")



## Configuration

Set `PROVIDED_CHIP_DIR` to the directory containing the chips you were given. Set `RAW_NAC_IMAGE_DIR` to the directory containing the original/base NAC GeoTIFFs. The raw directory is what gets indexed and queried by the Pipeline.


In [ ]:
DELETE_PREV_OUTPUTS = False
OUTPUT_DIR = Path("/explore/nobackup/people/ajkerr1/Lunar_FM/nac_verification_v2")
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

# Directory of provided chips to audit. Supports .nc, .tif, .tiff.
PROVIDED_CHIP_DIR = Path(
    "/explore/nobackup/projects/lfm/processed_data/Lunar/data_release/NAC_craters_coco_release_final/PHO"
)
PROVIDED_CHIP_GLOB = "*.nc"
NETCDF_VARIABLE = "band_data"

# Directory of raw/original NAC source GeoTIFFs.
RAW_NAC_IMAGE_DIR = Path(
    "/panfs/ccds02/nobackup/projects/lfm/processed_data/Lunar/LRO_NAC_Pho_Sites"
)
RAW_NAC_TILE_DB_NAME = "output_index.shp"

# IAU Moon geographic CRS used for lat/lon bbox queries.
IAU_30100_WKT_PATH = Path("/panfs/ccds02/nobackup/projects/lfm/IAU_30100_2015.wkt")

ZOOM_LEVEL = 5
MAX_SAMPLES = 10  # Set None to process all chips after smoke testing.
MAX_OVERLAP_CHECKS = 10

# If the provided product ID is not found in raw overlaps, process all raw overlaps.
# Set False to skip ambiguous/mismatched product IDs instead.
USE_ALL_OVERLAPS_WHEN_PRODUCT_MISSING = True

PLOTS_DIR = OUTPUT_DIR / "comparison_plots"
RECREATED_CHIP_DIR = OUTPUT_DIR / "raw_recreated_chips"
DATACUBE_DIR = OUTPUT_DIR / "datacubes"
METRICS_PATH = OUTPUT_DIR / "nac_verification_metrics.csv"
OVERLAP_SUMMARY_PATH = OUTPUT_DIR / "raw_overlap_summary.csv"

for path in [PLOTS_DIR, RECREATED_CHIP_DIR, DATACUBE_DIR]:
    path.mkdir(parents=True, exist_ok=True)

prep = prepare_output_dir(OUTPUT_DIR, DELETE_PREV_OUTPUTS)
print(f"OUTPUT_DIR: {OUTPUT_DIR}")



## Helper Functions


In [ ]:
MOON_SRS = (
    'GEOGCRS["Moon (2015) - Sphere / Ocentric",'
    'DATUM["Moon (2015) - Sphere",'
    'ELLIPSOID["Moon (2015) - Sphere",1737400,0,'
    'LENGTHUNIT["metre",1]]],'
    'PRIMEM["Reference Meridian",0,'
    'ANGLEUNIT["degree",0.0174532925199433]],'
    'CS[ellipsoidal,2],'
    'AXIS["geodetic latitude (Lat)",north,'
    'ORDER[1],'
    'ANGLEUNIT["degree",0.0174532925199433]],'
    'AXIS["geodetic longitude (Lon)",east,'
    'ORDER[2],'
    'ANGLEUNIT["degree",0.0174532925199433]],'
    'ID["IAU",30100,2015],'
    'REMARK["Source of IAU Coordinate systems: https://doi.org/10.1007/s10569-017-9805-5"]]'
)


def get_tile_db_path(image_dir, db_name="output_index.shp"):
    image_dir = Path(image_dir)
    if not image_dir.exists() or not image_dir.is_dir():
        raise ValueError(f"A valid image directory is required: {image_dir}")

    tile_db_path = image_dir / db_name
    if tile_db_path.exists():
        return tile_db_path

    tif_paths = sorted(image_dir.glob("*.tif"))
    if not tif_paths:
        raise FileNotFoundError(f"No .tif files found under {image_dir}")

    print(f"Creating tile index with {len(tif_paths)} raw NAC GeoTIFFs: {tile_db_path}")
    gdal.TileIndex(str(tile_db_path), [str(path) for path in tif_paths], outputSRS=MOON_SRS)
    tile_db_path.chmod(0o666)
    return tile_db_path


def product_id_from_path(path):
    stem = Path(path).stem
    match = re.search(r"M\d+[A-Z]+", stem)
    if match:
        return match.group(0)
    return stem.split("_")[0].split(".")[0]


def raw_product_id_from_path(path):
    return Path(path).stem.split(".")[0]


def open_chip(path, variable=NETCDF_VARIABLE):
    path = Path(path)
    if path.suffix.lower() == ".nc":
        return rxr.open_rasterio(f'NETCDF:"{path}":{variable}', masked=True)
    return rxr.open_rasterio(path, masked=True)


def read_chip_array(path, variable=NETCDF_VARIABLE):
    with open_chip(path, variable=variable) as da:
        arr = np.asarray(da.values)
    if arr.ndim == 3:
        arr = arr[0]
    return arr.astype("float64", copy=False)


def load_target_crs(wkt_path):
    with open(wkt_path, "r", encoding="utf-8") as f:
        return CRS.from_wkt(f.read())


def chip_lonlat_bounds(chip_path, target_crs):
    with open_chip(chip_path) as da:
        source_crs = da.rio.crs
        if source_crs is None:
            raise ValueError(f"No CRS found for {chip_path}")
        bounds = da.rio.bounds()

    bbox_geom = box(*bounds)
    transformer = Transformer.from_crs(source_crs, target_crs, always_xy=True)
    transformed = shapely_transform(transformer.transform, bbox_geom)
    min_lon, min_lat, max_lon, max_lat = transformed.bounds
    return (min_lon, min_lat, max_lon, max_lat)


def build_entry(chip_path, target_crs):
    chip_path = Path(chip_path)
    return {
        "location": chip_path,
        "filename": chip_path.name,
        "stem": chip_path.stem,
        "product_id": product_id_from_path(chip_path),
        "geometry": chip_lonlat_bounds(chip_path, target_crs),
    }


def collect_entries(chip_dir, chip_glob, target_crs, max_samples=None):
    chip_paths = sorted(Path(chip_dir).glob(chip_glob))
    if max_samples is not None:
        chip_paths = chip_paths[:max_samples]
    return [build_entry(path, target_crs) for path in tqdm(chip_paths, desc="Building entries")]



In [ ]:
def query_raw_nac_overlaps(geom_bounds, raw_tile_db_path, query_dir):
    ul_lon, lr_lat, lr_lon, ul_lat = geom_bounds
    query_dir = Path(query_dir)
    query_dir.mkdir(parents=True, exist_ok=True)

    pipeline = Pipeline(raw_tile_db_path, query_dir, debug=False, targetProductID=None)
    layer = pipeline._query(ul_lat, ul_lon, lr_lat, lr_lon)

    overlaps = []
    for feature in layer:
        raw_path = Path(feature["location"])
        overlaps.append(
            {
                "raw_path": raw_path,
                "raw_product_id": raw_product_id_from_path(raw_path),
            }
        )
    return overlaps


def summarize_raw_overlaps(entries, raw_tile_db_path):
    rows = []
    query_dir = OUTPUT_DIR / "_query_tmp"
    for entry in tqdm(entries, desc="Querying raw NAC overlaps"):
        overlaps = query_raw_nac_overlaps(entry["geometry"], raw_tile_db_path, query_dir)
        overlap_ids = sorted({item["raw_product_id"] for item in overlaps})
        rows.append(
            {
                "filename": entry["filename"],
                "provided_product_id": entry["product_id"],
                "num_raw_overlaps": len(overlaps),
                "num_raw_product_ids": len(overlap_ids),
                "provided_product_in_raw_overlaps": entry["product_id"] in overlap_ids,
                "raw_product_ids": ";".join(overlap_ids),
            }
        )
    return pd.DataFrame(rows)


def run_raw_nac_pipeline_for_entry(entry, raw_tile_db_path, datacube_dir, selected_product_id=None):
    ul_lon, lr_lat, lr_lon, ul_lat = entry["geometry"]
    datacube_dir = Path(datacube_dir)
    datacube_dir.mkdir(parents=True, exist_ok=True)

    cube_files = []
    status = "success"
    try:
        with redirect_stdout(sys.stderr):
            pipeline = Pipeline(
                raw_tile_db_path,
                datacube_dir,
                debug=False,
                targetProductID=selected_product_id,
            )
            tile_indexes = TmsIntersector().getTids(ul_lat, ul_lon, lr_lat, lr_lon, ZOOM_LEVEL)
            for idx in tile_indexes:
                tile_x = idx["tileX"]
                tile_y = idx["tileY"]
                zone = idx["zone"]
                tile_zoom = idx["zoomLevel"]
                tile_def = TmsTileDef.initFromParams(zone, tile_zoom)
                ulx, uly, lrx, lry = tile_def.getTileBbox(tile_x, tile_y)
                query_ul_lat, query_ul_lon = tile_def.ltmToLatLon(ulx, uly)
                query_lr_lat, query_lr_lon = tile_def.ltmToLatLon(lrx, lry)

                layer = pipeline._query(query_ul_lat, query_ul_lon, query_lr_lat, query_lr_lon)
                if layer.GetFeatureCount() == 0:
                    continue
                cube = pipeline._createCube(
                    layer,
                    ulx,
                    uly,
                    lrx,
                    lry,
                    tile_def.srs,
                    tile_def.tileWidth,
                    tile_def.tileHeight,
                    is_static=False,
                )
                if len(cube):
                    cube_files.extend(
                        pipeline._writeCube((tile_x, tile_y), cube, tile_def, ulx, uly)
                    )
        if not cube_files:
            status = "warning_no_cubes"
    except Exception as exc:
        print(f"Pipeline failed for {entry['filename']}: {exc}")
        status = "error_pipeline"
    return cube_files, status


def write_recreated_chip_from_datacubes(nac_files, reference_path, output_path):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    reference = open_chip(reference_path)
    nac_arrays = [rxr.open_rasterio(path, masked=True) for path in nac_files]
    try:
        merged = merge_arrays(nac_arrays, method="last")
        recreated = merged.rio.reproject_match(reference)
        if recreated.sizes.get("band", 1) > 1:
            recreated = recreated.isel(band=[0])
        recreated.rio.to_raster(output_path)
    finally:
        reference.close()
        for arr in nac_arrays:
            arr.close()
        if "merged" in locals():
            merged.close()
        if "recreated" in locals():
            recreated.close()
    return output_path



In [ ]:
def finite_pair_arrays(provided, recreated):
    provided = np.asarray(provided, dtype="float64")
    recreated = np.asarray(recreated, dtype="float64")
    mask = np.isfinite(provided) & np.isfinite(recreated)
    mask &= provided > -1e30
    mask &= recreated > -1e30
    return provided[mask], recreated[mask]


def gradient_sharpness(arr):
    arr = np.asarray(arr, dtype="float64")
    finite = np.isfinite(arr) & (arr > -1e30)
    if not finite.any():
        return np.nan
    fill = np.nanmedian(arr[finite])
    arr = np.where(finite, arr, fill)
    gy, gx = np.gradient(arr)
    return float(np.nanmean(np.sqrt(gx**2 + gy**2)))


def compare_arrays(provided, recreated):
    y_true, y_pred = finite_pair_arrays(provided, recreated)
    if y_true.size == 0:
        return {
            "valid_pixels": 0,
            "mae": np.nan,
            "rmse": np.nan,
            "corr": np.nan,
            "bias": np.nan,
        }
    diff = y_pred - y_true
    corr = np.corrcoef(y_true, y_pred)[0, 1] if y_true.size > 1 else np.nan
    return {
        "valid_pixels": int(y_true.size),
        "provided_min": float(np.nanmin(y_true)),
        "provided_max": float(np.nanmax(y_true)),
        "provided_mean": float(np.nanmean(y_true)),
        "provided_std": float(np.nanstd(y_true)),
        "recreated_min": float(np.nanmin(y_pred)),
        "recreated_max": float(np.nanmax(y_pred)),
        "recreated_mean": float(np.nanmean(y_pred)),
        "recreated_std": float(np.nanstd(y_pred)),
        "mae": float(np.nanmean(np.abs(diff))),
        "rmse": float(np.sqrt(np.nanmean(diff**2))),
        "bias": float(np.nanmean(diff)),
        "corr": float(corr),
        "provided_sharpness": gradient_sharpness(provided),
        "recreated_sharpness": gradient_sharpness(recreated),
    }


def percentile_stretch(arr, lower=2, upper=98):
    arr = np.asarray(arr, dtype="float64")
    finite = arr[np.isfinite(arr) & (arr > -1e30)]
    if finite.size == 0:
        return arr
    vmin, vmax = np.percentile(finite, [lower, upper])
    if vmax <= vmin:
        return arr
    return np.clip((arr - vmin) / (vmax - vmin), 0, 1)


def save_comparison_plot(entry, provided, recreated, output_path):
    diff = recreated - provided
    finite_diff = diff[np.isfinite(diff) & (provided > -1e30) & (recreated > -1e30)]
    if finite_diff.size:
        vmax = np.nanpercentile(np.abs(finite_diff), 98)
        norm = TwoSlopeNorm(vmin=-vmax, vcenter=0.0, vmax=vmax) if vmax > 0 else None
    else:
        norm = None

    fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
    axes[0].imshow(percentile_stretch(provided), cmap="gray")
    axes[0].set_title("Provided chip")
    axes[1].imshow(percentile_stretch(recreated), cmap="gray")
    axes[1].set_title("Raw-derived chip")
    im = axes[2].imshow(diff, cmap="coolwarm", norm=norm)
    axes[2].set_title("Raw - provided")
    fig.colorbar(im, ax=axes[2], fraction=0.046, pad=0.04)

    if finite_diff.size:
        axes[3].hist(finite_diff.ravel(), bins=80, color="0.25")
    axes[3].set_title("Difference histogram")

    for ax in axes[:3]:
        ax.axis("off")
    fig.suptitle(entry["filename"], fontsize=13, fontweight="bold")
    plt.tight_layout()
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return output_path



## Build Entries and Check Raw Overlaps

Run this before recreating chips. The overlap table tells you whether the product ID embedded in the provided chip filename is present in the raw NAC overlaps. If it is not present, the batch step will either use all raw overlaps or skip the sample, depending on `USE_ALL_OVERLAPS_WHEN_PRODUCT_MISSING`.


In [ ]:
target_crs = load_target_crs(IAU_30100_WKT_PATH)
raw_tile_db_path = get_tile_db_path(RAW_NAC_IMAGE_DIR, RAW_NAC_TILE_DB_NAME)
entries = collect_entries(PROVIDED_CHIP_DIR, PROVIDED_CHIP_GLOB, target_crs, MAX_SAMPLES)

print(f"Collected {len(entries)} provided chip entries")
print(entries[0] if entries else "No entries found")



In [ ]:
overlap_df = summarize_raw_overlaps(entries[:MAX_OVERLAP_CHECKS], raw_tile_db_path)
overlap_df.to_csv(OVERLAP_SUMMARY_PATH, index=False)
overlap_df



## Recreate One Chip from Raw NAC

Use this as a smoke test before running the batch loop. If product IDs do not match, this will show whether the all-overlaps fallback can produce a reasonable raw-derived chip.


In [ ]:
def select_raw_product_id(entry, overlaps):
    overlap_ids = sorted({item["raw_product_id"] for item in overlaps})
    if entry["product_id"] in overlap_ids:
        return entry["product_id"], "matched_product_id"
    if len(overlap_ids) == 1:
        return overlap_ids[0], "single_overlap_product_id"
    if USE_ALL_OVERLAPS_WHEN_PRODUCT_MISSING:
        return None, "all_overlaps_product_id_mismatch"
    return None, "skipped_product_id_mismatch"


def recreate_and_compare_entry(entry):
    sample_stem = entry["stem"]
    sample_datacube_dir = DATACUBE_DIR / sample_stem
    recreated_path = RECREATED_CHIP_DIR / f"{sample_stem}_raw_recreated.tif"
    plot_path = PLOTS_DIR / f"{sample_stem}_comparison.png"

    overlaps = query_raw_nac_overlaps(entry["geometry"], raw_tile_db_path, OUTPUT_DIR / "_query_tmp")
    selected_product_id, selection_status = select_raw_product_id(entry, overlaps)
    if selection_status == "skipped_product_id_mismatch":
        return {
            "filename": entry["filename"],
            "status": selection_status,
            "num_raw_overlaps": len(overlaps),
        }

    cube_files, pipeline_status = run_raw_nac_pipeline_for_entry(
        entry,
        raw_tile_db_path,
        sample_datacube_dir,
        selected_product_id=selected_product_id,
    )
    if pipeline_status != "success" or not cube_files:
        return {
            "filename": entry["filename"],
            "status": pipeline_status,
            "selection_status": selection_status,
            "selected_product_id": selected_product_id,
            "num_raw_overlaps": len(overlaps),
            "num_cube_files": len(cube_files),
        }

    write_recreated_chip_from_datacubes(cube_files, entry["location"], recreated_path)
    provided = read_chip_array(entry["location"])
    recreated = read_chip_array(recreated_path)
    metrics = compare_arrays(provided, recreated)
    save_comparison_plot(entry, provided, recreated, plot_path)

    row = {
        "filename": entry["filename"],
        "provided_product_id": entry["product_id"],
        "selected_product_id": selected_product_id,
        "selection_status": selection_status,
        "status": "success",
        "num_raw_overlaps": len(overlaps),
        "num_cube_files": len(cube_files),
        "recreated_path": str(recreated_path),
        "plot_path": str(plot_path),
    }
    row.update(metrics)
    return row



In [ ]:
single_result = recreate_and_compare_entry(entries[0])
single_result



## Batch Recreate and Compare

After the single-chip smoke test looks reasonable, set `MAX_SAMPLES = None` in the config and run this cell.


In [ ]:
rows = []
for entry in tqdm(entries, desc="Recreating and comparing NAC chips"):
    rows.append(recreate_and_compare_entry(entry))
    pd.DataFrame(rows).to_csv(METRICS_PATH, index=False)

metrics_df = pd.DataFrame(rows)
metrics_df.to_csv(METRICS_PATH, index=False)
print(f"Saved metrics: {METRICS_PATH}")
metrics_df.head()



In [ ]:
if "metrics_df" in globals() and len(metrics_df):
    display_cols = [
        "filename",
        "status",
        "selection_status",
        "num_raw_overlaps",
        "mae",
        "rmse",
        "corr",
        "provided_sharpness",
        "recreated_sharpness",
        "plot_path",
    ]
    display(metrics_df[[col for col in display_cols if col in metrics_df.columns]].sort_values("rmse", na_position="last").head(20))

